In [ ]:
!nvidia-smi

In [ ]:
!pip install -q librosa evaluate datasets jiwer gcsfs accelerate==0.27.2 peft==0.10.0 transformers==4.37.2

In [ ]:
!pip install -U -q datasets

In [ ]:
import librosa, torch, evaluate, os
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor, WhisperForConditionalGeneration, Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset
from dataclasses import dataclass
from typing import Any, Dict, List, Union

In [ ]:
print('Initializing...')

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-tiny")
tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-tiny", language="Bengali", task="transcribe")
processor = WhisperProcessor.from_pretrained("openai/whisper-tiny", language="Bengali", task="transcribe")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")

In [ ]:
def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    label_ids[label_ids == -100] = tokenizer.pad_token_id

    pred_str = tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    wer = 100 * wer_metric.compute(predictions=pred_str, references=label_str)

    return {"wer": wer}

In [ ]:
@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
def prepare_dataset(batch):
    audio_array, sampling_rate = librosa.load(batch["path"], sr=16000, mono=True)
    batch["input_features"] = feature_extractor(audio_array, sampling_rate=sampling_rate).input_features[0]
    batch["labels"] = tokenizer(batch["sentence"]).input_ids
    return batch

In [ ]:
ds = load_dataset("csv", data_files={"validation": ["rv2_valid_v2.csv"], "test": "test.csv"})
ds = ds.remove_columns(['gender', 'age', 'degree', 'state', 'district', 'area'])


In [ ]:
print(ds)
print(ds.column_names)
ds = ds.map(prepare_dataset, num_proc=None)
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="",  
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,  
    learning_rate=1e-5,
    warmup_steps=155,
    max_steps=3500,
    gradient_checkpointing=True,
    fp16=True,
    evaluation_strategy="steps",
    per_device_eval_batch_size=8,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=255,
    save_steps=500,
    eval_steps=500,
    logging_steps=1,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=False,

)


In [ ]:
processor.save_pretrained(training_args.output_dir)

In [ ]:
import random, torch, numpy as np
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)


In [ ]:
best_model_path = ""


In [ ]:
model = WhisperForConditionalGeneration.from_pretrained(best_model_path)
evaluator = Seq2SeqTrainer(args=training_args, model=model, data_collator=data_collator, compute_metrics=compute_metrics, tokenizer=processor.feature_extractor,)

In [ ]:
result = evaluator.evaluate(eval_dataset=ds["test"])
eval_wer = result["eval_wer"]

print("Test WER:", round(eval_wer, 4))

In [ ]:
# Get predictions
predictions = evaluator.predict(test_dataset=ds["test"], num_beams=5).predictions
predicted_transcriptions = processor.batch_decode(predictions, skip_special_tokens=True)
reference_transcriptions = ds["test"]["sentence"]

with open('', 'a', encoding='utf-8') as file_write:
    file_write.write('{}\t{}\t{}\t{}\t{}\n'.format('Sample', 'Prediction', 'Reference', 'WER', 'CER')) 

    for i, (pred, ref) in enumerate(zip(predicted_transcriptions, reference_transcriptions)):
        wer = 100 * wer_metric.compute(predictions=[pred], references=[ref])
        cer = 100 * cer_metric.compute(predictions=[pred], references=[ref])
        
        file_write.write('{}\t{}\t{}\t{}\t{}\n'.format(i+1, pred, ref, round(wer, 4), round(cer, 4)))
